In [2]:
import numpy as np
import pandas as pd
import math
import networkx as nx
import matplotlib.pyplot as plt
import random
from community import community_louvain  # python-louvain kütüphanesi gerekli
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score
import scipy.linalg as la
from statistics import mode
import numpy as np
import leidenalg
import igraph as ig
from collections import defaultdict
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score
from scipy.special import comb  # binom için
from collections import defaultdict
from networkx.algorithms.community import girvan_newman
from networkx.algorithms.community.centrality import girvan_newman
from itertools import islice


In [3]:

#df_combined= pd.read_csv('2022America.csv')
#df_combined= pd.read_csv("2022Asia.csv")
#df_combined= pd.read_csv('2022Europe.csv')



#df_combined= pd.read_csv('25usa.csv')
#df_combined=  pd.read_csv("25Asia.csv")
#df_combined= pd.read_csv('25Africa.csv')
#df_combined=  pd.read_csv('25europa.csv')
df_combined=  pd.read_csv('22all.csv')
#df_combined=  pd.read_csv('25all.csv')






In [4]:
df_combined

,0,Skill_1,Skill_2,Skill_3,Skill_4,Skill_5,Skill_6,Skill_7,Skill_8,Skill_9,...,Skill_41,Skill_42,Skill_43,Skill_44,Skill_45,Skill_46,Skill_47,Skill_48,Skill_49,Skill_50
0,Graphic Design\r\nAdobe Photoshop\r\nBanner Ad...,Graphic Design,Adobe Photoshop,Banner Ad Design,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Balance Sheet\r\nCash Flow Statement\r\nQuickb...,Balance Sheet,Cash Flow Statement,Quickbooks,Accounting Basics,Bookkeeping,Shopify,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Graphic Design\r\nWeb Development\r\nWeb Design,Graphic Design,Web Development,Web Design,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Adobe Photoshop Elements\r\nRAW\r\nAdobe Photo...,Adobe Photoshop Elements,RAW,Adobe Photoshop,Image Editing,Photo Manipulation,Photo Editing,People,Portrait Art,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Apache Airflow\r\nMySQL\r\nPython\r\nMachine L...,Apache Airflow,MySQL,Python,Machine Learning,RESTful API,Python Numpy FastAI,Deep Neural Network,Apache Solr,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,Spring Boot\r\nJava\r\nJavaScript,Spring Boot,Java,JavaScript,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14996,Animated Environment\r\n3D Sculpting\r\nAutode...,Animated Environment,3D Sculpting,Autodesk 3ds Max,Blender,Character Design,Animation,3D Animation,Motion Graphics,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14997,Video Editing\r\nAdobe Premiere Pro\r\nVideo P...,Video Editing,Adobe Premiere Pro,Video Post-Editing,Video Intro & Outro,YouTube Development,Video Production,Adobe After Effects,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14998,Network Security\r\nMalware,Network Security,Malware,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
from collections import Counter

# Kolon isimlerini string olarak kontrol et
skill_columns = [col for col in df_combined.columns if str(col).startswith("Skill_")]

# Tüm becerileri topla
all_skills = []
for col in skill_columns:
    all_skills.extend(df_combined[col].dropna().tolist())

# Sayım
skill_counts = Counter(all_skills)

# En çoktan en aza sırala
most_common_skills = skill_counts.most_common()

# Sonuçları yazdır
print("🔝 En sık geçen kalifikasyonlar:")
for skill, count in most_common_skills:
    print(f"{skill}: {count} kez")


🔝 En sık geçen kalifikasyonlar:
Graphic Design: 1288 kez
Content Writing: 1281 kez
English: 1160 kez
Web Development: 1103 kez
JavaScript: 1000 kez
Social Media Marketing: 967 kez
Web Design: 934 kez
WordPress: 850 kez
Researcher: 790 kez
Data Entry: 783 kez
Lead Generation: 744 kez
Search Engine Optimization: 639 kez
Adobe Photoshop: 619 kez
Facebook: 618 kez
Video Editing: 616 kez
HTML: 600 kez
CSS: 600 kez
Instagram: 588 kez
Copywriting: 585 kez
Virtual Assistant: 573 kez
Writing: 570 kez
PHP: 541 kez
Marketing Strategy: 525 kez
Android: 524 kez
Social Media Management: 513 kez
Adobe Illustrator: 494 kez
Creative Writing: 490 kez
Article Writing: 478 kez
Python: 448 kez
Video Production: 433 kez
Mobile App Development: 431 kez
API: 411 kez
Microsoft Excel: 399 kez
Communications: 398 kez
Facebook Advertising: 366 kez
Blog Content: 358 kez
SEO Writing: 351 kez
Translation: 350 kez
iOS: 347 kez
Android App Development: 346 kez
Logo Design: 339 kez
Shopify: 337 kez
Google Ads: 336 kez


In [10]:
Network = pd.DataFrame(most_common_skills, columns=["Skill", "Frekans"])

# Sonuç kontrolü
print(Network.head())

             Skill  Frekans
0   Graphic Design     1288
1  Content Writing     1281
2          English     1160
3  Web Development     1103
4       JavaScript     1000


In [12]:
Network["ID"]=range(len(Network))
Network

,Skill,Frekans,ID
0,Graphic Design,1288,0
1,Content Writing,1281,1
2,English,1160,2
3,Web Development,1103,3
4,JavaScript,1000,4
...,...,...,...
3801,Article Spinning,1,3801
3802,Green Card,1,3802
3803,Trade Law,1,3803
3804,Amicus Amicus Attorney,1,3804


In [14]:
# 1. Skill isimleriyle tam sayı (int) ID'leri eşleştiren sözlük oluştur
skill_to_id = dict(zip(Network["Skill"], Network["ID"].astype(int).astype(str)))

# 2. Skill kolonlarını seç
skill_columns = [col for col in df_combined.columns if str(col).startswith("Skill_")]

# 3. Her hücredeki beceriyi ID ile değiştir ve NaN yerine boş string ver
for col in skill_columns:
    df_combined[col] = df_combined[col].map(skill_to_id).fillna('').astype(str).replace('nan', '')

In [16]:
df_combined

,0,Skill_1,Skill_2,Skill_3,Skill_4,Skill_5,Skill_6,Skill_7,Skill_8,Skill_9,...,Skill_41,Skill_42,Skill_43,Skill_44,Skill_45,Skill_46,Skill_47,Skill_48,Skill_49,Skill_50
0,Graphic Design\r\nAdobe Photoshop\r\nBanner Ad...,0,12,274,,,,,,,...,,,,,,,,,,
1,Balance Sheet\r\nCash Flow Statement\r\nQuickb...,305,1128,391,316,136,41,,,,...,,,,,,,,,,
2,Graphic Design\r\nWeb Development\r\nWeb Design,0,3,6,,,,,,,...,,,,,,,,,,
3,Adobe Photoshop Elements\r\nRAW\r\nAdobe Photo...,1488,1121,12,156,271,78,327,744,,...,,,,,,,,,,
4,Apache Airflow\r\nMySQL\r\nPython\r\nMachine L...,2310,74,28,170,204,3548,2284,2463,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,Spring Boot\r\nJava\r\nJavaScript,504,53,4,,,,,,,...,,,,,,,,,,
14996,Animated Environment\r\n3D Sculpting\r\nAutode...,1116,1200,404,313,163,85,176,84,,...,,,,,,,,,,
14997,Video Editing\r\nAdobe Premiere Pro\r\nVideo P...,14,63,60,203,297,29,62,,,...,,,,,,,,,,
14998,Network Security\r\nMalware,280,2129,,,,,,,,...,,,,,,,,,,


In [18]:
# Skill_ kolonlarını seç
skill_columns = [col for col in df_combined.columns if str(col).startswith("Skill_")]

# Yeni bir kolon oluştur: Tüm Skill_ID'leri virgül ile birleştir
df_combined["Skill_ID_List"] = df_combined[skill_columns].apply(
    lambda row: ",".join([str(val) for val in row if val != '']), axis=1
)
# Sonucu göster
print(df_combined[["Skill_ID_List"]].head())

                       Skill_ID_List
0                           0,12,274
1            305,1128,391,316,136,41
2                              0,3,6
3    1488,1121,12,156,271,78,327,744
4  2310,74,28,170,204,3548,2284,2463


In [20]:
data = df_combined["Skill_ID_List"].dropna().tolist()

In [22]:
data= [row for row in data if any(c not in ' \t\n\r' for c in row)]
#data

In [24]:
# Kenar ağırlıklarını saklamak için sözlük
edge_weights = {}

for row in data:
    # Satırdaki düğümleri al (tekrarları ve boşlukları temizle)
    row_nodes = [int(node.strip()) for node in row.split(",")]
    
    # Tüm düğüm çiftleri için kenar oluştur (fully connected)
    for i in range(len(row_nodes)):
        for j in range(i + 1, len(row_nodes)):
            # Kenarı sıralı şekilde sakla (yönlü olmayan graf)
            edge = tuple(sorted((row_nodes[i], row_nodes[j])))
            
            # Kenar ağırlığını güncelle (varsa +1, yoksa 1 ata)
            if edge in edge_weights:
                edge_weights[edge] += 1
            else:
                edge_weights[edge] = 1

# Düğümleri bul (kenarlardan otomatik çıkar)
nodes = set()
for edge in edge_weights:
    nodes.add(edge[0])
    nodes.add(edge[1])

# NetworkX ağırlıklı graf oluştur
import networkx as nx
G = nx.Graph()

# Düğümleri ekle
G.add_nodes_from(nodes)

# Kenarları ve ağırlıklarını ekle
for edge, weight in edge_weights.items():
    G.add_edge(edge[0], edge[1], weight=weight)

# Graf bilgilerini yazdır
print("Düğümler:", G.nodes())
print("Kenarlar ve Ağırlıklar:")
for u, v, data in G.edges(data=True):
    print(f"{u}-{v}: weight={data['weight']}")


Düğümler: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219,

In [36]:
print(len(G.nodes))
print(len(G.edges))

3768
57994


In [26]:
from networkx.algorithms.community.quality import modularity
partition = community_louvain.best_partition(G,resolution=0.9, weight='weight')
#partition = community_louvain.best_partition(G, resolution=0.9)
kume_sayisi = max(partition.values()) + 1
modularity = community_louvain.modularity(partition, G, weight='weight')
print(f"\nOptimal Küme Sayısı: {kume_sayisi}")
print(f"Modularity Değeri: {modularity:.4f}")


Optimal Küme Sayısı: 24
Modularity Değeri: 0.6189


In [40]:
unique_communities = set(partition.values())
#print("Benzersiz Topluluklar:", unique_communities) 
# Toplulukları düğümlere göre grupla
community_to_nodes = defaultdict(list)

for node, community in partition.items():
    community_to_nodes[community].append(node)

# Her topluluğu ve düğümlerini yazdır
for comm, nodes in community_to_nodes.items():
    print(f"Topluluk {comm}: {nodes}")

Topluluk 0: [0, 12, 25, 40, 46, 78, 96, 97, 129, 133, 134, 137, 150, 156, 163, 175, 184, 201, 210, 227, 250, 271, 274, 286, 301, 309, 327, 335, 344, 345, 348, 354, 356, 364, 378, 387, 422, 426, 441, 455, 461, 481, 484, 494, 520, 550, 554, 564, 590, 611, 628, 635, 637, 639, 640, 653, 659, 669, 673, 676, 678, 681, 689, 691, 692, 743, 744, 746, 748, 754, 770, 779, 801, 816, 819, 826, 855, 856, 871, 873, 886, 899, 904, 917, 928, 937, 938, 962, 964, 966, 983, 989, 1002, 1003, 1009, 1024, 1037, 1038, 1041, 1051, 1068, 1078, 1080, 1086, 1089, 1109, 1119, 1121, 1130, 1139, 1147, 1199, 1207, 1210, 1212, 1228, 1237, 1259, 1278, 1307, 1315, 1339, 1346, 1350, 1351, 1352, 1360, 1379, 1395, 1397, 1402, 1425, 1437, 1446, 1452, 1454, 1455, 1461, 1463, 1464, 1480, 1488, 1493, 1508, 1510, 1511, 1515, 1524, 1526, 1528, 1562, 1573, 1584, 1587, 1598, 1601, 1603, 1612, 1616, 1623, 1635, 1638, 1649, 1650, 1664, 1668, 1680, 1683, 1685, 1690, 1707, 1743, 1753, 1759, 1776, 1785, 1788, 1791, 1794, 1796, 1801, 18

In [28]:
import leidenalg
from igraph import Graph as igraph_Graph

# NetworkX grafını igraph'a çevirme
G_igraph = igraph_Graph.from_networkx(G)  # G, nx.Graph() nesnesi olmalı

# Leiden uygula
partition = leidenalg.find_partition(G_igraph,  leidenalg.ModularityVertexPartition)
partition = {node: partition.membership[i] for i, node in enumerate(G.nodes())}


In [30]:
# Hangi düğüm hangi kümede?
print("\nDüğümlerin Toplulukları:")
for node, community in partition.items():
    print(f"Düğüm {node} → Küme {community}")
print("modularity:", community_louvain.modularity(partition, G, weight='weight'))



Düğümlerin Toplulukları:
Düğüm 0 → Küme 4
Düğüm 1 → Küme 3
Düğüm 2 → Küme 2
Düğüm 3 → Küme 0
Düğüm 4 → Küme 0
Düğüm 5 → Küme 3
Düğüm 6 → Küme 0
Düğüm 7 → Küme 0
Düğüm 8 → Küme 1
Düğüm 9 → Küme 1
Düğüm 10 → Küme 3
Düğüm 11 → Küme 3
Düğüm 12 → Küme 4
Düğüm 13 → Küme 3
Düğüm 14 → Küme 8
Düğüm 15 → Küme 0
Düğüm 16 → Küme 0
Düğüm 17 → Küme 3
Düğüm 18 → Küme 3
Düğüm 19 → Küme 1
Düğüm 20 → Küme 2
Düğüm 21 → Küme 0
Düğüm 22 → Küme 3
Düğüm 23 → Küme 0
Düğüm 24 → Küme 3
Düğüm 25 → Küme 4
Düğüm 26 → Küme 2
Düğüm 27 → Küme 3
Düğüm 28 → Küme 0
Düğüm 29 → Küme 8
Düğüm 30 → Küme 0
Düğüm 31 → Küme 0
Düğüm 32 → Küme 1
Düğüm 33 → Küme 1
Düğüm 34 → Küme 3
Düğüm 35 → Küme 3
Düğüm 36 → Küme 3
Düğüm 37 → Küme 2
Düğüm 38 → Küme 0
Düğüm 39 → Küme 0
Düğüm 40 → Küme 4
Düğüm 41 → Küme 1
Düğüm 42 → Küme 3
Düğüm 43 → Küme 1
Düğüm 44 → Küme 1
Düğüm 45 → Küme 3
Düğüm 46 → Küme 4
Düğüm 47 → Küme 1
Düğüm 48 → Küme 3
Düğüm 49 → Küme 1
Düğüm 50 → Küme 3
Düğüm 51 → Küme 0
Düğüm 52 → Küme 3
Düğüm 53 → Küme 0
Düğüm 54 → K

In [33]:
import leidenalg
from igraph import Graph as igraph_Graph
from collections import defaultdict
import networkx as nx  # modularity hesaplamak için gerekli

# NetworkX grafını igraph'a çevirme
G_igraph = igraph_Graph.from_networkx(G)  # G, nx.Graph() nesnesi olmalı

# Ağırlık ekleme (eğer varsa)
if 'weight' in G.edges():
    G_igraph.es['weight'] = [G[u][v]['weight'] for u, v in G.edges()]

# Resolution ayarıyla Leiden uygula (0.5-0.8 arası deneyin)
partition = leidenalg.find_partition(
    G_igraph, 
    leidenalg.RBConfigurationVertexPartition, 
    resolution_parameter=0.3,
    weights='weight' if 'weight' in G_igraph.edge_attributes() else None
)

# Yeni yöntemle küme bilgilerini al
communities = defaultdict(set)
for node_idx, community_id in enumerate(partition.membership):
    node_name = G_igraph.vs[node_idx]['_nx_name']  # Orijinal düğüm ismini al
    communities[community_id].add(node_name)

# Küme sayısı ve boyutları
print(f"Toplam küme sayısı: {len(communities)}")
for comm_id, nodes in communities.items():
    print(f"Küme {comm_id}: {len(nodes)} düğüm")

# Modularity hesapla ve yazdır
mod = partition.modularity
print(f"\nModularity değeri: {mod:.4f}")

# NetworkX ile modularity hesaplama alternatifi:
# nx_communities = [nodes for nodes in communities.values()]
# mod_nx = nx.algorithms.community.quality.modularity(G, nx_communities, weight='weight')
# print(f"NetworkX modularity: {mod_nx:.4f}")

Toplam küme sayısı: 15
Küme 2: 696 düğüm
Küme 0: 1816 düğüm
Küme 1: 1155 düğüm
Küme 3: 77 düğüm
Küme 4: 3 düğüm
Küme 6: 2 düğüm
Küme 7: 2 düğüm
Küme 5: 3 düğüm
Küme 10: 2 düğüm
Küme 11: 2 düğüm
Küme 12: 2 düğüm
Küme 13: 2 düğüm
Küme 8: 2 düğüm
Küme 14: 2 düğüm
Küme 9: 2 düğüm

Modularity değeri: 0.4210


In [35]:
# NetworkX ile modularity hesaplama alternatifi:
nx_communities = [nodes for nodes in communities.values()]
mod_nx = nx.algorithms.community.quality.modularity(G, nx_communities, weight='weight')
print(f"NetworkX modularity: {mod_nx:.4f}")

NetworkX modularity: 0.4876


In [37]:
subgraphs = []
for comm_id, nodes in communities.items():
    # Alt grafı oluştur (G'nin alt kümesi)
    subgraph = G.subgraph(nodes).copy()
    subgraphs.append(subgraph)
    
    # İsterseniz burada alt grafları kaydedebilir veya analiz edebilirsiniz
    print(f"Topluluk {comm_id} - Düğüm sayısı: {subgraph.number_of_nodes()}, Kenar sayısı: {subgraph.number_of_edges()}")

Topluluk 2 - Düğüm sayısı: 696, Kenar sayısı: 7062
Topluluk 0 - Düğüm sayısı: 1816, Kenar sayısı: 28182
Topluluk 1 - Düğüm sayısı: 1155, Kenar sayısı: 12571
Topluluk 3 - Düğüm sayısı: 77, Kenar sayısı: 426
Topluluk 4 - Düğüm sayısı: 3, Kenar sayısı: 2
Topluluk 6 - Düğüm sayısı: 2, Kenar sayısı: 1
Topluluk 7 - Düğüm sayısı: 2, Kenar sayısı: 1
Topluluk 5 - Düğüm sayısı: 3, Kenar sayısı: 3
Topluluk 10 - Düğüm sayısı: 2, Kenar sayısı: 1
Topluluk 11 - Düğüm sayısı: 2, Kenar sayısı: 1
Topluluk 12 - Düğüm sayısı: 2, Kenar sayısı: 1
Topluluk 13 - Düğüm sayısı: 2, Kenar sayısı: 1
Topluluk 8 - Düğüm sayısı: 2, Kenar sayısı: 1
Topluluk 14 - Düğüm sayısı: 2, Kenar sayısı: 1
Topluluk 9 - Düğüm sayısı: 2, Kenar sayısı: 1


In [42]:
import networkx as nx

# Leiden ile oluşturulmuş ilk alt ağı al
subgraphx = subgraphs[0]

# Tüm merkezilik ölçütlerini hesapla (ağırlıklı versiyonlar)
weight_param = 'weight' if 'weight' in subgraphx.edges() else None

centrality_metrics = {
    "Degree Centrality": nx.degree_centrality(subgraphx),
    "Betweenness Centrality": nx.betweenness_centrality(subgraphx, weight=weight_param),
    "Closeness Centrality": nx.closeness_centrality(subgraphx, distance=weight_param),
    "PageRank": nx.pagerank(subgraphx, weight=weight_param),
    "Authority (HITS)": nx.hits(subgraphx)[1],  # Sadece authority skorları
    "Eigenvector Centrality": nx.eigenvector_centrality(subgraphx, weight=weight_param, max_iter=1000)
}

# Her metrik için ilk 5 düğümü yazdır
for metric_name, scores in centrality_metrics.items():
    sorted_nodes = sorted(scores.items(), key=lambda x: -x[1])[:20]  # En yüksek 5 değer
    
    print(f"\n{metric_name} - Top 5:")
    for rank, (node, score) in enumerate(sorted_nodes, 1):
        print(f"{rank}. {node}: {[key for key, value in skill_to_id.items() if value == str(node)]}: {score:.4f}")

# Ekstra: Weighted Degree (Ağırlıklı derece) için
if weight_param:
    weighted_degrees = {node: sum(data['weight'] for _, _, data in subgraphx.edges(node, data=True)) 
                       for node in subgraphx.nodes()}
    
    print("\nWeighted Degree - Top 5:")
    for rank, (node, degree) in enumerate(sorted(weighted_degrees.items(), key=lambda x: -x[1])[:20], 1):
        print(f"{rank}. {node}: {degree:.2f}")


Degree Centrality - Top 5:
1. 0: ['Graphic Design']: 0.4791
2. 12: ['Adobe Photoshop']: 0.3568
3. 25: ['Adobe Illustrator']: 0.3381
4. 46: ['Illustration']: 0.2719
5. 72: ['3D Modeling']: 0.2158
6. 14: ['Video Editing']: 0.2101
7. 82: ['3D Design']: 0.1957
8. 40: ['Logo Design']: 0.1928
9. 91: ['3D Rendering']: 0.1813
10. 29: ['Video Production']: 0.1741
11. 62: ['Adobe After Effects']: 0.1640
12. 85: ['Animation']: 0.1583
13. 78: ['Photo Editing']: 0.1554
14. 137: ['Infographic']: 0.1540
15. 163: ['Character Design']: 0.1410
16. 129: ['Layout Design']: 0.1367
17. 134: ['Adobe InDesign']: 0.1338
18. 96: ['Social Media Imagery']: 0.1324
19. 132: ['Architectural Design']: 0.1324
20. 161: ['Autodesk AutoCAD']: 0.1309

Betweenness Centrality - Top 5:
1. 0: ['Graphic Design']: 0.2576
2. 12: ['Adobe Photoshop']: 0.1147
3. 25: ['Adobe Illustrator']: 0.0918
4. 72: ['3D Modeling']: 0.0807
5. 14: ['Video Editing']: 0.0591
6. 46: ['Illustration']: 0.0561
7. 82: ['3D Design']: 0.0526
8. 161: ['Au